# Apply the Cleaning Decisions

`04_cleaning_decisions_explained.ipynb` recommended an action for each data problem found in `03_data_cleaning_review.ipynb`. This notebook **carries out every recommended action**, one decision at a time, and checks each one with an assertion.

## Natural-language request

> Take all the cleaning decisions that are recommended and create another notebook where those decisions are implemented.

## What gets implemented

| Notebook 04 step | Decision | Notebook 03 | What this notebook does |
|---|---|---|---|
| 1–5 | Keep repeated, zero-night, zero-guest, room `P`, 10+ adult and ADR-outlier rows | C5 | Adds ten `flag_` columns; removes no row |
| 6 | `meal = Undefined` | C1 | Recodes `Undefined` to `SC` |
| 7 | NULL `agent` / `company`, decimal IDs | C2 | Stores IDs as text; NULL becomes `none` |
| 8 | Missing `country`; `CN` vs `CHN` | C3 | Labels missing as `unknown`; keeps `CN` and `CHN` separate |
| 9 | 4 missing `children` | C4 | Fills with 0; `flag_children_missing` records which rows |
| 10 | Checks needing no action | — | Asserts those columns are unchanged |
| 11 | `previous_cancellations` drift | C6 | Changes no value; writes down the two input sets Step 5 must compare on choose |

## Ground rules (same as notebooks 03 and 04)

- The raw file is never changed; all work is on a copy.
- No cleaning step reads `is_canceled`, `reservation_status` or `reservation_status_date`. Every flag and recode uses structural columns only, so applying them to choose and proof reveals no proof outcome.
- No row is removed, and the contract's scoring inputs (`hotel`, `arrival_date`, `adr`, `nights`) stay identical.
- `src/contract.py` is used unchanged.

## Setup

Same loading code as notebooks 03 and 04. `nights` is the contract's definition; `guests` is used only for the validity flags.

In [1]:
from pathlib import Path
import sys

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "data.py").exists():
        repo_root = folder
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")

source_folder = str(repo_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

import contract
import data
import pandas as pd

raw_bookings = data.read_raw()
source_columns = list(raw_bookings.columns)
outcome_columns = ["is_canceled", "reservation_status", "reservation_status_date"]
structural_columns = [column for column in source_columns if column not in outcome_columns]

bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
bookings["guests"] = bookings["adults"] + bookings["children"].fillna(0) + bookings["babies"]

# The working copy every decision below is applied to.
cleaned = bookings.copy()


def count_by_period(df, mask):
    """Count rows where mask is True in each contract period (no outcome column is read)."""
    train, choose, proof = contract.split_periods(df.assign(_mask=mask))
    return {name: int(period["_mask"].sum()) for name, period in
            {"train": train, "choose": choose, "proof": proof}.items()}


print(f"Rows: {len(raw_bookings):,}; source columns: {len(source_columns)}")

Rows: 119,390; source columns: 32


## The decisions being applied

All six notebook 03 proposals are set to `"approve"`, because each is the recommendation in notebook 04. To reverse one, change it to `"disapprove"` and re-run: its step will then change nothing and say so.

In [2]:
cleaning_decisions = {
    "C1_meal_undefined_to_sc": "approve",
    "C2_agent_company_text_with_none": "approve",
    "C3_country_unknown_label": "approve",
    "C4_children_fill_zero": "approve",
    "C5_keep_all_rows_with_flags": "approve",
    "C6_test_prior_cancellations_on_choose": "approve",
}

invalid = {key: value for key, value in cleaning_decisions.items() if value not in {"approve", "disapprove"}}
if invalid:
    raise ValueError(f"Use approve or disapprove: {invalid}")
print(pd.Series(cleaning_decisions).to_string())

C1_meal_undefined_to_sc                  approve
C2_agent_company_text_with_none          approve
C3_country_unknown_label                 approve
C4_children_fill_zero                    approve
C5_keep_all_rows_with_flags              approve
C6_test_prior_cancellations_on_choose    approve


## Decision 1 (C5): keep every row and flag the odd ones

Covers notebook 04 steps 1–5. A flag records an observation and changes nothing. The flags are added **first**, before any recode, so that `flag_children_missing` captures the 4 rows before C4 fills them.

| Flag | Notebook 04 step | Marks |
|---|---|---|
| `flag_identical_group` | 1 | Rows identical to another row in every non-outcome column |
| `flag_zero_nights` | 2 | Same-day bookings (`nights = 0`) |
| `flag_zero_guests`, `flag_room_type_p`, `flag_children_without_adults` | 3 | Missing or odd guest counts; placeholder room type |
| `flag_ten_plus_adults` | 4 | 10 or more adults on one row |
| `flag_adr_above_1000`, `flag_adr_negative` | 5 | The ADR 5,400 and -6.38 rows |
| `flag_children_missing` | 9 | The 4 rows whose `children` is filled by C4 |
| `flag_repeat_status_conflict` | 11 | Not a repeated guest, yet has prior bookings |

In [3]:
rows_before = len(cleaned)

if cleaning_decisions["C5_keep_all_rows_with_flags"] == "approve":
    cleaned["flag_identical_group"] = cleaned[structural_columns].duplicated(keep=False)  # no outcome column read
    cleaned["flag_zero_nights"] = cleaned["nights"].eq(0)
    cleaned["flag_zero_guests"] = cleaned["guests"].eq(0)
    cleaned["flag_room_type_p"] = cleaned["reserved_room_type"].eq("P")
    cleaned["flag_children_without_adults"] = cleaned["adults"].eq(0) & cleaned["guests"].gt(0)
    cleaned["flag_ten_plus_adults"] = cleaned["adults"].ge(10)
    cleaned["flag_adr_above_1000"] = cleaned["adr"].gt(1000)  # only the 5,400 row; next highest is 508
    cleaned["flag_adr_negative"] = cleaned["adr"].lt(0)
    cleaned["flag_children_missing"] = cleaned["children"].isna()
    cleaned["flag_repeat_status_conflict"] = cleaned["is_repeated_guest"].eq(0) & (
        cleaned["previous_cancellations"].gt(0) | cleaned["previous_bookings_not_canceled"].gt(0)
    )
else:
    print("C5 disapproved: no flags added. Row removal is paused until a rule is agreed.")

flag_columns = [column for column in cleaned.columns if column.startswith("flag_")]

# Check: flagging removed no row and changed no source value.
assert len(cleaned) == rows_before == len(raw_bookings)
assert cleaned[source_columns].equals(raw_bookings)

flag_counts = pd.DataFrame({column: count_by_period(cleaned, cleaned[column]) for column in flag_columns}).T
print(f"Rows kept: {len(cleaned):,} of {len(raw_bookings):,}; source values unchanged.\n")
print("Flagged rows by period:")
print(flag_counts.to_string())

Rows kept: 119,390 of 119,390; source values unchanged.

Flagged rows by period:
                              train  choose  proof
flag_identical_group          19982    8856  11916
flag_zero_nights                371     200    144
flag_zero_guests                 49      64     67
flag_room_type_p                  1       5      6
flag_children_without_adults     72      88     63
flag_ten_plus_adults             13       0      0
flag_adr_above_1000               1       0      0
flag_adr_negative                 0       0      1
flag_children_missing             4       0      0
flag_repeat_status_conflict    5571     445    221


## Decision 2 (C1): recode `meal = Undefined` to `SC`

The dataset article defines `Undefined` and `SC` as the same "no meal package" category (notebook 04, step 6).

In [4]:
undefined_before = int(cleaned["meal"].eq("Undefined").sum())
sc_before = int(cleaned["meal"].eq("SC").sum())

if cleaning_decisions["C1_meal_undefined_to_sc"] == "approve":
    cleaned["meal"] = cleaned["meal"].replace("Undefined", "SC")

    # Check: every Undefined became SC, and nothing else moved.
    assert cleaned["meal"].eq("Undefined").sum() == 0
    assert cleaned["meal"].eq("SC").sum() == sc_before + undefined_before
    other_codes = ~raw_bookings["meal"].isin(["Undefined", "SC"])
    assert cleaned.loc[other_codes, "meal"].equals(raw_bookings.loc[other_codes, "meal"])
    print(f"Recoded {undefined_before:,} rows from Undefined to SC; SC count {sc_before:,} -> {sc_before + undefined_before:,}")
else:
    print("C1 disapproved: meal left unchanged.")

print("\nMeal codes after:", cleaned["meal"].value_counts().to_dict())

Recoded 1,169 rows from Undefined to SC; SC count 10,650 -> 11,819

Meal codes after: {'BB': 92310, 'HB': 14463, 'SC': 11819, 'FB': 798}


## Decision 3 (C2): store `agent` and `company` IDs as text; NULL becomes `none`

The article says NULL here means "not applicable" (no agent, no company), and the IDs are labels, not quantities (notebook 04, step 7).

In [5]:
if cleaning_decisions["C2_agent_company_text_with_none"] == "approve":
    for column in ["agent", "company"]:
        null_before = int(raw_bookings[column].isna().sum())
        # Every ID is a whole number, so 9.0 -> "9" loses nothing.
        cleaned[column] = raw_bookings[column].map(lambda value: "none" if pd.isna(value) else str(int(value)))

        # Check: NULLs became "none", no value is missing, and no ID kept a decimal point.
        assert cleaned[column].eq("none").sum() == null_before
        assert cleaned[column].notna().all()
        assert not cleaned[column].str.contains(".", regex=False).any()
        print(f"{column}: {null_before:,} NULL -> 'none'; {cleaned[column].nunique() - 1} distinct IDs now text "
              f"(for example {cleaned.loc[cleaned[column].ne('none'), column].iloc[0]!r})")
else:
    print("C2 disapproved: agent and company left as decimals with NaN.")

agent: 16,340 NULL -> 'none'; 333 distinct IDs now text (for example '304')


company: 112,593 NULL -> 'none'; 352 distinct IDs now text (for example '110')


## Decision 4 (C3): label missing `country` as `unknown`; keep `CN` and `CHN` separate

`country` is Bin 3 and never a predictor. `CN` is probably China, but no source confirms it, so it is not merged (notebook 04, step 8).

In [6]:
missing_before = int(cleaned["country"].isna().sum())

if cleaning_decisions["C3_country_unknown_label"] == "approve":
    cleaned["country"] = cleaned["country"].fillna("unknown")

    # Check: only the missing values changed; CN and CHN are still separate.
    assert cleaned["country"].eq("unknown").sum() == missing_before
    known = raw_bookings["country"].notna()
    assert cleaned.loc[known, "country"].equals(raw_bookings.loc[known, "country"])
    print(f"Labelled {missing_before:,} missing countries as 'unknown'")
else:
    print("C3 disapproved: missing country left as NaN.")

print(f"CN rows: {cleaned['country'].eq('CN').sum():,}; CHN rows: {cleaned['country'].eq('CHN').sum():,} (not merged)")

Labelled 488 missing countries as 'unknown'
CN rows: 1,279; CHN rows: 999 (not merged)


## Decision 5 (C4): fill the 4 missing `children` with 0

95.1% of train rows have 0 children. `flag_children_missing` (added in Decision 1) records which rows were filled (notebook 04, step 9).

In [7]:
missing_rows = cleaned.index[cleaned["children"].isna()]

if cleaning_decisions["C4_children_fill_zero"] == "approve":
    cleaned["children"] = cleaned["children"].fillna(0).astype("int64")

    # Check: exactly the flagged rows were filled, and the column is a whole-number count again.
    assert len(missing_rows) == 4
    assert cleaned.loc[missing_rows, "children"].eq(0).all()
    if "flag_children_missing" in cleaned:
        assert cleaned.index[cleaned["flag_children_missing"]].equals(missing_rows)
    assert cleaned["children"].dtype == "int64"
    print(f"Filled {len(missing_rows)} rows with 0 (index {list(missing_rows)}); children dtype now {cleaned['children'].dtype}")
else:
    print("C4 disapproved: children left with 4 NaN values.")

Filled 4 rows with 0 (index [40600, 40667, 40679, 41160]); children dtype now int64


## Decision 6 (C6): no change to `previous_cancellations`; test it on choose

This is a feature decision, not a value fix (notebook 04, step 11). Implementing it here means two things:

1. Assert that the column's values are unchanged.
2. Write down the two input sets that Step 5 must compare on **choose**, so the test cannot be forgotten. The winning set is frozen before the single proof run (rule 2).

In [8]:
candidate_inputs_with = ["hotel", "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled"]
candidate_inputs_without = [column for column in candidate_inputs_with if column != "previous_cancellations"]

# Check: no candidate input was changed by cleaning.
assert cleaned[candidate_inputs_with].equals(raw_bookings[candidate_inputs_with])

if cleaning_decisions["C6_test_prior_cancellations_on_choose"] == "approve":
    step5_input_sets = {"with previous_cancellations": candidate_inputs_with,
                        "without previous_cancellations": candidate_inputs_without}
    print("Step 5 must score every candidate on choose with each input set:")
    for name, columns in step5_input_sets.items():
        print(f"  {name}: {columns}")
else:
    step5_input_sets = {"with previous_cancellations": candidate_inputs_with}
    print("C6 disapproved: the feature decision is paused; only the full input set is recorded.")

print("\nCandidate input values unchanged.")

Step 5 must score every candidate on choose with each input set:
  with previous_cancellations: ['hotel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled']
  without previous_cancellations: ['hotel', 'is_repeated_guest', 'previous_bookings_not_canceled']

Candidate input values unchanged.


## Final checks on the cleaned table

These assertions are the proof that the decisions were applied as recommended and nothing else changed:

- every row is kept and the raw table is untouched;
- the only source columns whose values changed are the five that C1–C4 recode;
- outcome columns, candidate inputs and the contract's scoring inputs are identical;
- each hotel-night's book size and stay value V is identical in every period (computed without any outcome column).

In [9]:
expected_changed = {"meal", "agent", "company", "country", "children"}

# 1. Rows and raw table.
assert len(cleaned) == len(raw_bookings)
assert bookings[source_columns].equals(raw_bookings)

# 2. Only the recoded columns differ from the raw values.
changed = {column for column in source_columns if not cleaned[column].equals(raw_bookings[column])}
assert changed == expected_changed, changed

# 3. Outcomes, candidate inputs and scoring inputs are untouched.
scoring_inputs = ["hotel", "arrival_date", "adr", "nights"]
assert cleaned[outcome_columns].equals(raw_bookings[outcome_columns])
assert cleaned[candidate_inputs_with].equals(raw_bookings[candidate_inputs_with])
assert cleaned[scoring_inputs].equals(bookings[scoring_inputs])


# 4. Every hotel-night's book and V are identical (structural columns only).
def night_book(df):
    stay = df["adr"] * df["nights"]
    return stay.groupby([df["hotel"], df["arrival_date"]]).agg(["size", "mean"])


assert night_book(cleaned).equals(night_book(bookings))

print("All checks passed.")
print(f"  rows kept: {len(cleaned):,}; columns: {cleaned.shape[1]} "
      f"({len(source_columns)} source + arrival_date, nights, guests + {len(flag_columns)} flags)")
print(f"  source columns changed: {sorted(changed)}")
print(f"  hotel-nights with identical book and V: {len(night_book(cleaned)):,}")

All checks passed.
  rows kept: 119,390; columns: 45 (32 source + arrival_date, nights, guests + 10 flags)
  source columns changed: ['agent', 'children', 'company', 'country', 'meal']
  hotel-nights with identical book and V: 1,586


## Split the cleaned table into the contract periods

The cleaned table uses the same arrival-date split as everything else. Only row counts are shown; no outcome is read here.

In [10]:
clean_train, clean_choose, clean_proof = contract.split_periods(cleaned)
print("Rows per period:", {"train": len(clean_train), "choose": len(clean_choose), "proof": len(clean_proof)})
print("\nColumn types of the changed columns:")
print(cleaned[sorted(expected_changed)].dtypes.to_string())

Rows per period: {'train': 49157, 'choose': 29546, 'proof': 40687}

Column types of the changed columns:
agent         str
children    int64
company       str
country       str
meal          str


## Save the cleaned table

Saved twice, both in `data/`. The `data/` folder is in `.gitignore`, so neither file is committed; re-run this notebook to rebuild them.

- `hotels_clean.pkl` keeps the column types (`arrival_date` as a date, IDs as text, flags as True/False). Load it with `pd.read_pickle(...)`. Use this one in later notebooks.
- `hotels_clean.csv` is the same table for opening in Excel or other tools. A CSV stores no types, so when reading it with pandas pass `dtype={"agent": str, "company": str}`, `parse_dates=["arrival_date"]` and `keep_default_na=False` (otherwise country code `NA`, Namibia, would be read as missing).

In [11]:
clean_path = repo_root / "data" / "hotels_clean.pkl"
cleaned.to_pickle(clean_path)

# Check: the saved file reloads identically.
assert pd.read_pickle(clean_path).equals(cleaned)
print(f"Saved {len(cleaned):,} rows x {cleaned.shape[1]} columns to {clean_path.relative_to(repo_root)}")

csv_path = repo_root / "data" / "hotels_clean.csv"
cleaned.to_csv(csv_path, index=False)

# Check: the CSV reloads with the same rows, columns and values.
csv_reloaded = pd.read_csv(csv_path, dtype={"agent": str, "company": str},
                           parse_dates=["arrival_date"], keep_default_na=False)
assert csv_reloaded.shape == cleaned.shape
assert list(csv_reloaded.columns) == list(cleaned.columns)
pd.testing.assert_frame_equal(csv_reloaded, cleaned, check_dtype=False)
print(f"Saved {len(cleaned):,} rows x {cleaned.shape[1]} columns to {csv_path.relative_to(repo_root)}")

Saved 119,390 rows x 45 columns to data\hotels_clean.pkl


Saved 119,390 rows x 45 columns to data\hotels_clean.csv


## Rows for `docs/DECISIONS.md`

`src/data.py` asks that every cleaning choice be logged with its reason and row count. The cell below builds those rows from the counts computed above, ready to paste.

In [12]:
log_rows = [
    ("C5: keep every row; add flag_ columns",
     "No booking ID proves any row is an error; removal would change the contract's book and the train rate (35.8% -> 22.2%)",
     f"0 removed; {len(flag_columns)} flag columns added"),
    ("C1: meal Undefined -> SC", "Article: 'Undefined/SC - no meal package'", f"{undefined_before:,} recoded"),
    ("C2: agent/company as text; NULL -> none", "Article: NULL means not applicable; IDs are labels",
     f"agent {raw_bookings['agent'].isna().sum():,}, company {raw_bookings['company'].isna().sum():,} labelled none; all IDs retyped"),
    ("C3: missing country -> unknown; CN kept separate", "Bin 3, never a predictor; CN meaning unconfirmed",
     f"{missing_before:,} labelled"),
    ("C4: missing children -> 0, with flag", "95.1% of train rows have 0; 4 rows in one incomplete cluster",
     f"{len(missing_rows)} filled"),
    ("C6: previous_cancellations unchanged; test with/without on choose",
     "Share with prior cancellations falls 11.8% (train) -> 1.3% (choose)", "0 changed"),
]

print("| Date | Choice | Evidence / reason | Rows touched | Status |")
print("|---|---|---|---:|---|")
for choice, reason, rows in log_rows:
    key = next(k for k in cleaning_decisions if k.startswith(choice[:2]))
    status = "Applied" if cleaning_decisions[key] == "approve" else "Paused"
    print(f"| {pd.Timestamp.today().date()} | {choice} | {reason} | {rows} | {status} |")

| Date | Choice | Evidence / reason | Rows touched | Status |
|---|---|---|---:|---|
| 2026-11-18 | C5: keep every row; add flag_ columns | No booking ID proves any row is an error; removal would change the contract's book and the train rate (35.8% -> 22.2%) | 0 removed; 10 flag columns added | Applied |
| 2026-11-18 | C1: meal Undefined -> SC | Article: 'Undefined/SC - no meal package' | 1,169 recoded | Applied |
| 2026-11-18 | C2: agent/company as text; NULL -> none | Article: NULL means not applicable; IDs are labels | agent 16,340, company 112,593 labelled none; all IDs retyped | Applied |
| 2026-11-18 | C3: missing country -> unknown; CN kept separate | Bin 3, never a predictor; CN meaning unconfirmed | 488 labelled | Applied |
| 2026-11-18 | C4: missing children -> 0, with flag | 95.1% of train rows have 0; 4 rows in one incomplete cluster | 4 filled | Applied |
| 2026-11-18 | C6: previous_cancellations unchanged; test with/without on choose | Share with prior cancellations falls

## Summary

- **All six recommended decisions are applied.** C1–C4 recode five columns (`meal`, `agent`, `company`, `country`, `children`). C5 adds ten `flag_` columns. C6 changes no value and records the with/without input sets for Step 5.
- **Nothing else changed.** Every row is kept; outcome columns, candidate inputs and the contract's scoring inputs are identical; every hotel-night's book and V are the same as in the raw data. The assertions above fail loudly if any of this stops being true.
- **No outcome was read** by any cleaning step, so the proof period remains sealed.

**Next:**

1. Paste the rows above into `docs/DECISIONS.md`.
2. In Step 5, score every candidate on choose with both input sets in `step5_input_sets`, and freeze the choice before proof.
3. Still open after cleaning (from notebook 04): group bookings cancel together, so the contract's `spread` understates uncertainty when choosing the buffer; whether same-day stays belong in the book is a client question.